<a href="https://colab.research.google.com/github/incrisvel/adaptaste/blob/main/similarity_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install pandas numpy scipy scikit-learn plotly gensim transformers

In [2]:
import pandas as pd
import json

# Abre arquivo de reviews
with open('reviews_preprocessed.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

preprocessed_reviews_df = pd.DataFrame(data)

In [3]:
preprocessed_reviews_df.head()

,review_id,review_date,place_id,review_text,rating,description_clean,description_tokenized,description_normalized,description_no_stopwords,description_stemmed,description_lemmatized,description_bigrams
0,Ci9DQUlRQUNvZENodHljRjlvT2xaVlRqSXpPRTFSYURKdl...,2026-07-30T17:09:02+00:00,0x6b0a883715bb4c6b:0,Excelente restaurante! A comida é muito saboro...,5.0,Excelente restaurante! A comida é muito saboro...,"[Excelente, restaurante, !, A, comida, é, muit...","[excelente, restaurante, a, comida, é, muito, ...","[excelente, restaurante, comida, muito, saboro...","[excel, restaurant, com, muit, sabor, bem, ser...","[excelente, restaurante, comer, muito, saboros...","[[excelente, restaurante], [restaurante, comer..."
1,Ci9DQUlRQUNvZENodHljRjlvT20xVFVVaHpaRVV4VVZWSl...,2026-05-31T17:09:02+00:00,0x6b0a883715bb4c6b:0,Fui pela primeira vez e gostei bastante! A com...,5.0,Fui pela primeira vez e gostei bastante! A com...,"[Fui, pela, primeira, vez, e, gostei, bastante...","[fui, pela, primeira, vez, e, gostei, bastante...","[primeira, vez, gostei, bastante, comida, muit...","[prim, vez, gost, bast, com, muit, sabor, peg,...","[primeiro, vez, gostar, bastante, comer, muito...","[[primeiro, vez], [vez, gostar], [gostar, bast..."
2,Ci9DQUlRQUNvZENodHljRjlvT2psSmFEVkVNak5NTnpONV...,2026-05-31T17:09:02+00:00,0x6b0a883715bb4c6b:0,Ambiente simples e gostoso familiar comida óti...,5.0,Ambiente simples e gostoso familiar comida óti...,"[Ambiente, simples, e, gostoso, familiar, comi...","[ambiente, simples, e, gostoso, familiar, comi...","[ambiente, simples, gostoso, familiar, comida,...","[ambi, simpl, gost, famili, com, ótim, gost, m...","[ambiente, simples, gostoso, familiar, comer, ...","[[ambiente, simples], [simples, gostoso], [gos..."
3,ChZDSUhNMG9nS0VJQ0FnSUNHMWZEb1NBEAE,2022-08-30T17:10:53+00:00,0x6b0a883715bb4c6b:0,"Se é qualidade que procura, não tem outra igua...",5.0,"Se é qualidade que procura, não tem outra igua...","[Se, é, qualidade, que, procura, ,, não, tem, ...","[se, é, qualidade, que, procura, não, tem, out...","[qualidade, procura, não, outra, igual, agilid...","[qual, proc, não, outr, igual, agil, entreg, t...","[qualidade, procurar, não, outro, igual, agili...","[[qualidade, procurar], [procurar, não], [não,..."
4,ChdDSUhNMG9nS0VJQ0FnSUNHc2NtMjRBRRAB,2022-08-30T17:10:52+00:00,0x6b0a883715bb4c6b:0,"Comida maravilhosa. Super caseira, bem tempera...",5.0,"Comida maravilhosa. Super caseira, bem tempera...","[Comida, maravilhosa, ., Super, caseira, ,, be...","[comida, maravilhosa, super, caseira, bem, tem...","[comida, maravilhosa, super, caseira, bem, tem...","[com, maravilh, sup, cas, bem, temper, tend, s...","[comida, maravilhoso, super, caseira, bem, tem...","[[comida, maravilhoso], [maravilhoso, super], ..."


# Bag of Words

In [4]:
from sklearn.feature_extraction.text import CountVectorizer

# Extrai todos os tokens de todas as reviews
tokens_column = preprocessed_reviews_df["description_no_stopwords"]
tokens = tokens_column.tolist()

bow_vectorizer = CountVectorizer(
    analyzer=lambda tokens: tokens
)
bow_matrix = bow_vectorizer.fit_transform(
    tokens_column
)

# Recupera o vocabulário
bow_words = bow_vectorizer.get_feature_names_out()

# Transforma a matriz em DataFrame
df_bow = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_words
)

# Adicionar o ID para conectar com a base original
df_bow.insert(0, "review_text", preprocessed_reviews_df["review_text"].values)

# Visualizar
df_bow.head()

,review_text,aaaaamei,abacaxi,abaixo,abendessen,abençoada,abençoe,aber,abertas,aberto,...,ótima,ótimas,ótimo,ótimos,última,últimas,último,única,único,über
0,Excelente restaurante! A comida é muito saboro...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Fui pela primeira vez e gostei bastante! A com...,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Ambiente simples e gostoso familiar comida óti...,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,"Se é qualidade que procura, não tem outra igua...",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,"Comida maravilhosa. Super caseira, bem tempera...",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# TF-IDF

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Criar o modelo TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    analyzer=lambda tokens: tokens
)

# Criar matriz
tfidf_matrix = tfidf_vectorizer.fit_transform(
    preprocessed_reviews_df["description_no_stopwords"]
)

# Recuperar o vocabulário
tfidf_words = tfidf_vectorizer.get_feature_names_out()

# Transformar matriz em DF
df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_words
)

df_tfidf.insert(0, "review_text", preprocessed_reviews_df["review_text"].values)

df_tfidf.head(20)

,review_text,aaaaamei,abacaxi,abaixo,abendessen,abençoada,abençoe,aber,abertas,aberto,...,ótima,ótimas,ótimo,ótimos,última,últimas,último,única,único,über
0,Excelente restaurante! A comida é muito saboro...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,Fui pela primeira vez e gostei bastante! A com...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
2,Ambiente simples e gostoso familiar comida óti...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.151180,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,"Se é qualidade que procura, não tem outra igua...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,"Comida maravilhosa. Super caseira, bem tempera...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
5,As palavras para descrever a comida é simplesm...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
6,"A comida deve ser boa pelas indicações mas, o ...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
7,Excelente comida e serviço,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
8,Top,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
9,Não sei na outras empresas mas já foi servido...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


# Word2Vec

In [6]:
from gensim.models import Word2Vec
import numpy as np

tokens_w2v = preprocessed_reviews_df["description_no_stopwords"]

w2v_model = Word2Vec(
    sentences=tokens_w2v,
    vector_size=190,
    window=5,
    min_count=1,
    sg=1,
    seed=42,
    workers=1,
    epochs=200
)

def w2v_document_vector(tokens):
    # Mesma função de semantic_search.ipynb: média dos vetores conhecidos.
    vectors = [
        w2v_model.wv[token]
        for token in tokens
        if token in w2v_model.wv
    ]

    if not vectors:
        return np.zeros(w2v_model.vector_size, dtype=np.float32)

    return np.mean(vectors, axis=0)

w2v_matrix = np.vstack(
    [w2v_document_vector(t) for t in tokens_w2v]
)

# BERT

In [7]:
import torch
from transformers import AutoTokenizer, AutoModel

BERT_MODEL = "neuralmind/bert-base-portuguese-cased"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_bert = AutoTokenizer.from_pretrained(BERT_MODEL, do_lower_case=False)
bert_model = AutoModel.from_pretrained(BERT_MODEL).to(device)
bert_model.eval()

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(
        last_hidden_state.size()
    ).float()

    total = torch.sum(last_hidden_state * mask, dim=1)

    return total / torch.clamp(mask.sum(dim=1), min=1e-9)


def generate_bert_embeddings(texts, batch_size=8):
    embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        inputs = tokenizer_bert(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            output = bert_model(**inputs)

        embeddings.append(
            mean_pooling(
                output.last_hidden_state,
                inputs["attention_mask"]
            ).cpu().numpy()
        )

    return np.vstack(embeddings)

original_texts = preprocessed_reviews_df["review_text"].tolist()
bert_matrix = generate_bert_embeddings(original_texts)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Similaridade entre textos

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
# Calcula similaridade a partir do BoW
similarity_bow = cosine_similarity(
    df_bow.drop(columns="review_text")
)

dt_sim_bow = pd.DataFrame(
    similarity_bow,
    index=preprocessed_reviews_df["review_text"],
    columns=preprocessed_reviews_df["review_text"]
)

dt_sim_bow.head()

review_text,"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨","Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!","Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau","As palavras para descrever a comida é simplesmente SENSACIONAL. Muito sabor, tudo muito fresco, além de ser muito gostoso o que eles fazem. RECOMENDADÍSSIMO. Podiam trazer uma franquia dessa para São Paulo.\nA próxima vez que estivermos passeando em Blumenau com certeza traremos mais. Uma pena não ter pego mais.","A comida deve ser boa pelas indicações mas, o telefone é de enfeite pois ninguém atende, nem mesmo após 20 ligações em um espaço de 30 minutos!!",Excelente comida e serviço,Top,"Não sei na outras empresas mas já foi servido frango crú ou duro , macarrão branco, comida sem sal algum a feijoada não e boa.",...,muito saborosa,Excelente!,Sensacional,A melhor da região ❤️❤️,Excelente,Perfeito ❤️🌹❤️ …,A melhor de todas! Recomendo,Tudo ótimo e atendimento excelente!,Marmitas deliciosas 😋🤩 …,Recomendo!!! Super deliciosas!!\n😋 😋 …
review_text,,,,,,,,,,,,,,,,,,,,,
"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨",1.000000,0.227110,0.183533,0.223607,0.127920,0.300000,0.051640,0.230940,0.0,0.044721,...,0.424264,0.2,0.0,0.0,0.2,0.0,0.000000,0.200000,0.0,0.000000
"Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",0.227110,1.000000,0.186081,0.108821,0.138343,0.189258,0.041885,0.093659,0.0,0.072548,...,0.229416,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.093659
Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,0.183533,0.186081,1.000000,0.153897,0.097823,0.191180,0.059235,0.132453,0.0,0.051299,...,0.162221,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000
"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!",0.223607,0.108821,0.153897,1.000000,0.000000,0.372678,0.000000,0.000000,0.0,0.100000,...,0.316228,0.0,0.0,0.0,0.0,0.0,0.000000,0.111803,0.0,0.000000
"Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau",0.127920,0.138343,0.097823,0.000000,1.000000,0.106600,0.110096,0.246183,0.0,0.095346,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.123091,0.000000,0.0,0.369274


In [12]:
# Calcula similaridade a partir do TF-IDF
similarity_tfidf = cosine_similarity(
    df_tfidf.drop(columns="review_text")
)

dt_sim_tfidf = pd.DataFrame(
    similarity_tfidf,
    index=preprocessed_reviews_df["review_text"],
    columns=preprocessed_reviews_df["review_text"]
)

dt_sim_tfidf.head()

review_text,"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨","Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!","Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau","As palavras para descrever a comida é simplesmente SENSACIONAL. Muito sabor, tudo muito fresco, além de ser muito gostoso o que eles fazem. RECOMENDADÍSSIMO. Podiam trazer uma franquia dessa para São Paulo.\nA próxima vez que estivermos passeando em Blumenau com certeza traremos mais. Uma pena não ter pego mais.","A comida deve ser boa pelas indicações mas, o telefone é de enfeite pois ninguém atende, nem mesmo após 20 ligações em um espaço de 30 minutos!!",Excelente comida e serviço,Top,"Não sei na outras empresas mas já foi servido frango crú ou duro , macarrão branco, comida sem sal algum a feijoada não e boa.",...,muito saborosa,Excelente!,Sensacional,A melhor da região ❤️❤️,Excelente,Perfeito ❤️🌹❤️ …,A melhor de todas! Recomendo,Tudo ótimo e atendimento excelente!,Marmitas deliciosas 😋🤩 …,Recomendo!!! Super deliciosas!!\n😋 😋 …
review_text,,,,,,,,,,,,,,,,,,,,,
"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨",1.000000,0.101567,0.046544,0.074669,0.041217,0.123324,0.008559,0.087817,0.0,0.007912,...,0.261253,0.122276,0.0,0.0,0.122276,0.0,0.000000,0.101837,0.0,0.000000
"Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",0.101567,1.000000,0.080314,0.035230,0.045367,0.080605,0.005258,0.020171,0.0,0.024956,...,0.125176,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.052753
Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,0.046544,0.080314,1.000000,0.063365,0.015287,0.055853,0.007381,0.028316,0.0,0.006823,...,0.049590,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000
"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!",0.074669,0.035230,0.063365,1.000000,0.000000,0.185659,0.000000,0.000000,0.0,0.049793,...,0.117975,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.107273,0.0,0.000000
"Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau",0.041217,0.045367,0.015287,0.000000,1.000000,0.031610,0.016993,0.065189,0.0,0.015709,...,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.070189,0.000

In [13]:
similarity_w2v = cosine_similarity(w2v_matrix)

dt_sim_w2v = pd.DataFrame(
    similarity_w2v,
    index=preprocessed_reviews_df["review_text"],
    columns=preprocessed_reviews_df["review_text"]
)

dt_sim_w2v.head()

review_text,"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨","Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!","Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau","As palavras para descrever a comida é simplesmente SENSACIONAL. Muito sabor, tudo muito fresco, além de ser muito gostoso o que eles fazem. RECOMENDADÍSSIMO. Podiam trazer uma franquia dessa para São Paulo.\nA próxima vez que estivermos passeando em Blumenau com certeza traremos mais. Uma pena não ter pego mais.","A comida deve ser boa pelas indicações mas, o telefone é de enfeite pois ninguém atende, nem mesmo após 20 ligações em um espaço de 30 minutos!!",Excelente comida e serviço,Top,"Não sei na outras empresas mas já foi servido frango crú ou duro , macarrão branco, comida sem sal algum a feijoada não e boa.",...,muito saborosa,Excelente!,Sensacional,A melhor da região ❤️❤️,Excelente,Perfeito ❤️🌹❤️ …,A melhor de todas! Recomendo,Tudo ótimo e atendimento excelente!,Marmitas deliciosas 😋🤩 …,Recomendo!!! Super deliciosas!!\n😋 😋 …
review_text,,,,,,,,,,,,,,,,,,,,,
"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨",1.000000,0.744583,0.658925,0.728807,0.698058,0.761934,0.581311,0.575875,0.336542,0.587539,...,0.625406,0.401434,0.336673,0.418244,0.401434,0.409550,0.552581,0.617035,0.441168,0.521964
"Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",0.744583,1.000000,0.721856,0.660118,0.697822,0.714823,0.642588,0.513945,0.338902,0.603034,...,0.479672,0.281433,0.298665,0.459424,0.281433,0.324750,0.557402,0.488363,0.490777,0.583473
Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,0.658925,0.721856,1.000000,0.649644,0.645147,0.635498,0.485272,0.456212,0.281349,0.531734,...,0.473739,0.270823,0.189163,0.357016,0.270823,0.268628,0.422982,0.462881,0.443525,0.465798
"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!",0.728807,0.660118,0.649644,1.000000,0.586233,0.796711,0.562771,0.461060,0.292040,0.602853,...,0.500173,0.325570,0.335423,0.411212,0.325570,0.401207,0.519658,0.597974,0.459189,0.510590
"Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau",0.698058,0.697822,0.645147,0.586233,1.0000

In [14]:
similarity_bert = cosine_similarity(bert_matrix)

dt_sim_bert = pd.DataFrame(
    similarity_bert,
    index=preprocessed_reviews_df["review_text"],
    columns=preprocessed_reviews_df["review_text"]
)

dt_sim_bert.head()

review_text,"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨","Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!","Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau","As palavras para descrever a comida é simplesmente SENSACIONAL. Muito sabor, tudo muito fresco, além de ser muito gostoso o que eles fazem. RECOMENDADÍSSIMO. Podiam trazer uma franquia dessa para São Paulo.\nA próxima vez que estivermos passeando em Blumenau com certeza traremos mais. Uma pena não ter pego mais.","A comida deve ser boa pelas indicações mas, o telefone é de enfeite pois ninguém atende, nem mesmo após 20 ligações em um espaço de 30 minutos!!",Excelente comida e serviço,Top,"Não sei na outras empresas mas já foi servido frango crú ou duro , macarrão branco, comida sem sal algum a feijoada não e boa.",...,muito saborosa,Excelente!,Sensacional,A melhor da região ❤️❤️,Excelente,Perfeito ❤️🌹❤️ …,A melhor de todas! Recomendo,Tudo ótimo e atendimento excelente!,Marmitas deliciosas 😋🤩 …,Recomendo!!! Super deliciosas!!\n😋 😋 …
review_text,,,,,,,,,,,,,,,,,,,,,
"Excelente restaurante! A comida é muito saborosa, bem servida e dá para perceber a qualidade dos ingredientes. O atendimento também foi muito atencioso e o ambiente é agradável. Com certeza é um lugar que vale a pena conhecer e que pretendo voltar! 🍜✨",1.000000,0.872569,0.748484,0.857366,0.818004,0.862910,0.763952,0.630993,0.128488,0.696677,...,0.506250,0.458745,0.371510,0.513898,0.371957,0.383809,0.624099,0.672702,0.455325,0.680623
"Fui pela primeira vez e gostei bastante! A comida é muito saborosa. Pegamos as opções sem pimenta/menos picantes por sugestão de amigos e valeu super a pena - mesmo ficando com curiosidade de experimentar os pratos apimentados. Os atendentes são uns queridos e o ambiente, apesar de simples, é aconchegante - com músicas coreanas tocando em volume agradável e decorações. Além disso eles tem estacionamento próprio.",0.872569,1.000000,0.758789,0.836514,0.842065,0.843108,0.805649,0.614330,0.186847,0.770783,...,0.527543,0.426962,0.373023,0.511762,0.359582,0.345303,0.590785,0.621869,0.520492,0.645455
Ambiente simples e gostoso familiar comida ótima gostamos muito moramos em Barra Velha e fomos comemorar dia das mães com nossa filha gostamos bastante,0.748484,0.758789,1.000000,0.736001,0.717713,0.719118,0.667214,0.583134,0.167392,0.660455,...,0.481364,0.390256,0.333026,0.494451,0.349141,0.333121,0.523584,0.618636,0.419409,0.591483
"Se é qualidade que procura, não tem outra igual.\nAgilidade na entrega e tudo muito gostoso, opções de sabores, gosto muito da refeição dessa equipe.\nParabéns!",0.857366,0.836514,0.736001,1.000000,0.819232,0.830746,0.750651,0.677641,0.232465,0.750533,...,0.597985,0.523340,0.425589,0.562324,0.456429,0.415930,0.607076,0.715731,0.536068,0.678399
"Comida maravilhosa. Super caseira, bem temperada mesmo tendo sido congelada, depois de aquecer tem gostinho de comida feita na hora. Super recomendo para as pessoas de Blumenau",0.818004,0.842065,0.717713,0.819232,1.0000

# Heatmap de similaridade

In [82]:
import plotly.graph_objects as go

texts = preprocessed_reviews_df["review_text"]

def create_heatmap(matrix, title):
  custom_data = []

  for i in range(len(texts)):
    line = []

    for j in range(len(texts)):
      line.append([
          texts[i],
          texts[j]
      ])

    custom_data.append(line)

  fig = go.Figure(
      data=go.Heatmap(
          z=matrix,
          x=texts,
          y=texts,
          colorscale="Blues",
          zmin=0,
          zmax=1,
          customdata=custom_data,
          hovertemplate=
            "<b>%{customdata[0]}</b><br>"
                "<b>%{customdata[1]}</b><br>"
                "<b>Similaridade:</b> %{z:.3f}"
                "<extra></extra>"
      )
  )

  fig.update_layout(
      title=title,
      xaxis_title="Review",
      yaxis_title="Review",
      width=900,
      height=850
  )

  fig.show()

In [ ]:
create_heatmap(
    similarity_bow,
    "Similaridade entre reviews — Bag of Words"
)

In [ ]:
create_heatmap(
    similarity_tfidf,
    "Similaridade entre reviews — TF-IDF"
)

In [ ]:
create_heatmap(
    similarity_w2v,
    "Similaridade entre reviews — Word2Vec"
)

In [ ]:
create_heatmap(
    similarity_bert,
    "Similaridade entre reviews — Bert"
)

# Clustering

In [15]:
model_data = {
    "BoW": {
        "matrix": bow_matrix,
        "terms": bow_words,
        "similarity": similarity_bow,
        "models_history": {},  # Todos os K-Means gerados
        "metrics_df": None,    # DataFrame com as inércias e silhouettes
        "k": None,
        "labels": None,
        "model": None,
        "distance_centroids": None
    },
    "TF-IDF": {
        "matrix": tfidf_matrix,
        "terms": tfidf_words,
        "similarity": similarity_tfidf,
        "models_history": {},
        "metrics_df": None,
        "k": None,
        "labels": None,
        "model": None,
        "distance_centroids": None
    },
   "Word2Vec": {
        "matrix": w2v_matrix,
        "terms": None,
        "similarity": similarity_w2v,
        "models_history": {},
        "metrics_df": None,
        "k": None,
        "labels": None,
        "model": None,
        "distance_centroids": None
    },
    "BERT": {
        "matrix": bert_matrix,
        "terms": None,
        "similarity": similarity_bert,
        "models_history": {},
        "metrics_df": None,
        "k": None,
        "labels": None,
        "model": None,
        "distance_centroids": None
    }
}

In [16]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

N_CLUSTERS_TEST = 20

def execute_cluster_demos(matrix, model_name):
    n_samples = matrix.shape[0]

    k_values = [k for k in range(2, min(N_CLUSTERS_TEST+1, n_samples))]

    results = []
    models = {}

    for k in k_values:
        model = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10,
            max_iter=300
        )

        labels = model.fit_predict(matrix)
        n_labels = len(np.unique(labels))

        if 1 < n_labels < n_samples:
            mat_eval = matrix.toarray() if hasattr(matrix, "toarray") else matrix

            silhouette = silhouette_score(
                mat_eval,
                labels,
                metric="cosine",
                sample_size=min(2000, n_samples),
                random_state=42
            )
        else:
            silhouette = np.nan

        results.append({
            "k": k,
            "inertia": model.inertia_,
            "silhouette_cosine": silhouette,
            "cluster_sizes": np.bincount(labels).tolist()
        })

        models[k] = model

    return models, pd.DataFrame(results)

In [17]:
# Extrair as métricas do K-Means até N_CLUSTERS_TEST clusters
for model_name, data in model_data.items():
    matrix = data["matrix"]

    models_history, metrics_df = execute_cluster_demos(matrix, model_name)

    data["models_history"] = models_history
    data["metrics_df"] = metrics_df

In [18]:
import plotly.express as px

for model_name, data in model_data.items():

    # Plotar o Método do Cotovelo (Inércia)
    fig = px.line(
        data["metrics_df"],
        x="k",
        y="inertia",
        markers=True,
        title=f"Método do cotovelo - {model_name} + K-Means",
        labels={
            "k": "Número de clusters (K)",
            "inertia": "Inércia"
        }
    )
    fig.show()

In [19]:
for model_name, data in model_data.items():
    print("-" * 100)
    print(model_name)
    print("-" * 100)
    display(
        data["metrics_df"].sort_values(
            "silhouette_cosine",
            ascending=False
        )
    )

----------------------------------------------------------------------------------------------------
BoW
----------------------------------------------------------------------------------------------------


,k,inertia,silhouette_cosine,cluster_sizes
9,11,16355.423629,0.115268,"[46, 947, 3, 252, 1, 115, 289, 251, 2, 3, 353]"
17,19,15366.442939,0.094464,"[116, 360, 1, 124, 811, 3, 1, 76, 1, 1, 247, 4..."
18,20,15367.077851,0.066091,"[14, 1017, 101, 1, 1, 10, 1, 524, 6, 1, 106, 2..."
0,2,18605.222375,0.045039,"[1777, 485]"
3,5,17377.799697,0.044131,"[1519, 281, 458, 3, 1]"
1,3,18190.137641,0.042507,"[587, 1558, 117]"
16,18,15359.595488,0.040859,"[28, 108, 1, 205, 1, 1, 1016, 167, 3, 1, 1, 1,..."
10,12,16195.470121,0.040812,"[371, 1312, 63, 1, 212, 1, 286, 1, 1, 1, 1, 12]"
14,16,15812.111079,0.037734,"[21, 1, 347, 14, 133, 1, 183, 1, 222, 109, 105..."
8,10,16110.635694,0.034218,"[46, 1, 16, 277, 1, 352, 15, 1174, 378, 2]"


----------------------------------------------------------------------------------------------------
TF-IDF
----------------------------------------------------------------------------------------------------


,k,inertia,silhouette_cosine,cluster_sizes
16,18,1614.098513,0.193443,"[17, 69, 104, 102, 86, 78, 89, 69, 82, 23, 36,..."
17,19,1600.535605,0.189572,"[52, 212, 90, 101, 139, 97, 52, 40, 68, 76, 49..."
15,17,1620.200846,0.183964,"[24, 100, 69, 81, 67, 40, 99, 35, 143, 85, 40,..."
14,16,1648.066107,0.177933,"[48, 1094, 100, 105, 146, 52, 26, 69, 87, 207,..."
11,13,1676.752168,0.173758,"[49, 104, 99, 69, 1298, 110, 108, 88, 35, 40, ..."
10,12,1699.456570,0.172096,"[1168, 154, 120, 92, 102, 101, 184, 69, 110, 5..."
12,14,1674.194655,0.168749,"[51, 145, 66, 181, 90, 110, 102, 89, 109, 1061..."
18,20,1596.983168,0.167524,"[630, 52, 87, 104, 89, 35, 98, 77, 47, 267, 39..."
13,15,1646.304387,0.163981,"[264, 997, 40, 91, 67, 168, 50, 152, 49, 58, 1..."
9,11,1717.191575,0.160781,"[263, 1250, 101, 52, 39, 104, 159, 82, 107, 36..."


----------------------------------------------------------------------------------------------------
Word2Vec
----------------------------------------------------------------------------------------------------


,k,inertia,silhouette_cosine,cluster_sizes
18,20,14115.708008,0.215914,"[306, 83, 22, 65, 150, 50, 75, 64, 63, 103, 95..."
14,16,15227.008789,0.215001,"[323, 85, 65, 504, 25, 57, 97, 469, 22, 84, 10..."
10,12,16794.517578,0.213611,"[216, 65, 80, 89, 1024, 146, 122, 321, 107, 22..."
7,9,17990.033203,0.207204,"[136, 1286, 65, 108, 89, 22, 147, 351, 58]"
8,10,17532.650391,0.206723,"[1122, 108, 363, 117, 91, 22, 100, 57, 65, 217]"
6,8,18792.857422,0.205905,"[147, 1292, 92, 108, 136, 77, 65, 345]"
11,13,16165.393555,0.205756,"[108, 85, 124, 34, 268, 152, 102, 75, 58, 1077..."
16,18,14622.789062,0.202477,"[109, 83, 86, 100, 183, 33, 92, 22, 65, 64, 25..."
13,15,15536.197266,0.201436,"[289, 930, 83, 105, 108, 92, 72, 27, 25, 34, 9..."
4,6,20721.021484,0.198855,"[1700, 110, 97, 65, 136, 154]"


----------------------------------------------------------------------------------------------------
BERT
----------------------------------------------------------------------------------------------------


,k,inertia,silhouette_cosine,cluster_sizes
1,3,43147.230469,0.301341,"[127, 1323, 812]"
0,2,46342.527344,0.279215,"[842, 1420]"
3,5,38068.339844,0.226731,"[552, 241, 127, 552, 790]"
2,4,40384.128906,0.208471,"[775, 718, 652, 117]"
5,7,35780.550781,0.198112,"[108, 449, 567, 208, 413, 314, 203]"
6,8,34874.476562,0.196661,"[416, 550, 190, 443, 176, 73, 99, 315]"
4,6,36727.031250,0.193174,"[485, 574, 240, 129, 414, 420]"
9,11,32498.166016,0.186178,"[58, 245, 248, 297, 188, 70, 29, 416, 182, 99,..."
8,10,33457.035156,0.183830,"[184, 327, 109, 428, 310, 58, 98, 508, 73, 167]"
11,13,31499.236328,0.179289,"[196, 185, 196, 390, 401, 55, 195, 214, 29, 18..."


In [20]:
# Selecionar o melhor modelo

model_data["BoW"]["k"] = 10
model_data["TF-IDF"]["k"] = 15
model_data["Word2Vec"]["k"] = 9
model_data["BERT"]["k"] = 11

for model_name, data in model_data.items():
    chosen_k = data["k"]

    if chosen_k is not None:
        final_model = data["models_history"][chosen_k]

        data["final_labels"] = final_model.labels_
        data["model"] = final_model

        col_name = f"cluster_{model_name.lower().replace('-', '_')}"
        preprocessed_reviews_df[col_name] = final_model.labels_

# Análise de clustering


## Verificação manual


In [21]:
TARGET_MODEL = "BERT"

cluster_column = f"cluster_{TARGET_MODEL.lower().replace('-', '_')}"

view_columns = [
    "review_id",
    "place_id",
    "rating",
    "review_text",
    cluster_column
]

display(preprocessed_reviews_df[view_columns].sort_values(by=cluster_column))

,review_id,place_id,rating,review_text,cluster_bert
876,ChdDSUhNMG9nS0VJQ0FnSUNBallDSm5BRRAB,0x94df192bbbd5d5bb:0,4.0,Melhor steak house de Blumenau! Excelente cart...,0
873,ChdDSUhNMG9nS0VJQ0FnSURnaUp1TnJnRRAB,0x94df192bbbd5d5bb:0,5.0,Uma das melhores opções em Blumenau,0
852,ChdDSUhNMG9nS0VJQ0FnSUNBaDdfcTFRRRAB,0x94df192bbbd5d5bb:0,5.0,Bestes Steakhaus in Blumenau,0
2255,ChZDSUhNMG9nS0VJQ0FnSUM3aC11ckhREAE,0x94df3b76767a0235:0,5.0,A melhor da região ❤️❤️,0
840,ChZDSUhNMG9nS0VJQ0FnSURza1pEZWN3EAE,0x94df192bbbd5d5bb:0,5.0,O melhor de Blumenau 😋 …,0
...,...,...,...,...,...
896,Ci9DQUlRQUNvZENodHljRjlvT2xkeVVHbFZjRWxwVG5Vd1...,0x94df192d88bb4431:0,5.0,Adorei. Várias opções e ambiente aconchegante.,10
19,Ci9DQUlRQUNvZENodHljRjlvT2xwMWRFbHdOazF2V0VsbW...,0x94df1891f8b8e1fd:0,5.0,"Comida caseira, bem feita. Lugar tranquilo, li...",10
20,Ci9DQUlRQUNvZENodHljRjlvT25od05rRnpVblpFVjFoMV...,0x94df1891f8b8e1fd:0,5.0,Sou frequentador assiduo. Super recomendo. Um ...,10
2244,Ci9DQUlRQUNvZENodHljRjlvT205U2MyWnJiVWRuVjFGNk...,0x94df3b76767a0235:0,5.0,"Comida deliciosa, chega rápido, e com muito ca...",10


## PCA

In [22]:
from sklearn.decomposition import PCA

for model_name, data in model_data.items():
    print("-" * 100)
    print(model_name)
    print("-" * 100)

    # Reduzir o TF-IDF para duas dimensões
    pca_2d = PCA(n_components=2)

    matrix = data["matrix"]

    coordinates_2d = pca_2d.fit_transform(
        matrix.toarray() if hasattr(matrix, "toarray") else matrix
    )

    cluster_column = f"cluster_{model_name.lower().replace('-', '_')}"

    # Criar um DataFrame para a visualização
    df_pca_2d = pd.DataFrame({
        "id": preprocessed_reviews_df["review_id"],
        "text": preprocessed_reviews_df["review_text"],
        "place": preprocessed_reviews_df["place_id"],
        "cluster": preprocessed_reviews_df[cluster_column].astype(str),
        "PCA1": coordinates_2d[:, 0],
        "PCA2": coordinates_2d[:, 1]
    })

    # Criar o gráfico
    fig = px.scatter(
        df_pca_2d,
        x="PCA1",
        y="PCA2",
        color="cluster",
        hover_name="text",
        hover_data={
            "place": True,
            "PCA1": False,
            "PCA2": False
        },
        title=f"{data["k"]} clusters em 2D ({model_name}) - Variância: {pca_2d.explained_variance_ratio_.sum():.3f}"
    )

    fig.update_traces(marker=dict(size=8, opacity=0.7))

    fig.update_layout(
        xaxis_title="Componente principal 1",
        yaxis_title="Componente principal 2",
        legend_title="Cluster"
    )

    fig.show()

----------------------------------------------------------------------------------------------------
BoW
----------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------
TF-IDF
----------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------
Word2Vec
----------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------
BERT
----------------------------------------------------------------------------------------------------


In [23]:
for model_name, data in model_data.items():
    print("-" * 100)
    print(model_name)
    print("-" * 100)

    # Reduzir o TF-IDF para três dimensões
    pca_3d = PCA(n_components=3)

    matrix = data["matrix"]

    coordinates_3d = pca_3d.fit_transform(
        matrix.toarray() if hasattr(matrix, "toarray") else matrix
    )

    cluster_column = f"cluster_{model_name.lower().replace('-', '_')}"

    # Criar DataFrame para visualização
    df_pca_3d = pd.DataFrame({
        "id": preprocessed_reviews_df["review_id"],
        "text": preprocessed_reviews_df["review_text"],
        "place": preprocessed_reviews_df["place_id"],
        "cluster": preprocessed_reviews_df[cluster_column].astype(str),
        "PCA1": coordinates_3d[:, 0],
        "PCA2": coordinates_3d[:, 1],
        "PCA3": coordinates_3d[:, 2]
    })

    # Criar gráfico 3D
    fig = px.scatter_3d(
        df_pca_3d,
        x="PCA1",
        y="PCA2",
        z="PCA3",
        color="cluster",
        hover_name="text",
        hover_data={
            "place": True,
            "PCA1": False,
            "PCA2": False,
            "PCA3": False
        },
        title=f"Clusters em 3D ({model_name}) - Variância: {pca_3d.explained_variance_ratio_.sum():.3f}"
    )

    fig.update_traces(marker=dict(size=8, opacity=0.7))

    fig.update_layout(
        scene=dict(
            xaxis_title="Componente principal 1",
            yaxis_title="Componente principal 2",
            zaxis_title="Componente principal 3"
        ),
        legend_title="Cluster"
    )

    fig.show()

----------------------------------------------------------------------------------------------------
BoW
----------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------
TF-IDF
----------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------
Word2Vec
----------------------------------------------------------------------------------------------------


----------------------------------------------------------------------------------------------------
BERT
----------------------------------------------------------------------------------------------------


## Termos representativos

In [24]:
for model_name, data in model_data.items():
    print("-" * 100)
    print(model_name)
    print("-" * 100)

    if data["terms"] is None:
      print("N/A")
      continue

    # Centroides calculados pelo K-Means
    centroids = data["model"].cluster_centers_

    # Quantidade de termos que queremos mostrar
    n_terms = 10

    results_clusters = []

    for cluster_id in range(data["k"]):

        # Ordenar os termos pelo peso no centroide
        indices = np.argsort(
            centroids[cluster_id]
        )[::-1]

        principal_indices = indices[:n_terms]

        principal_terms = data["terms"][principal_indices]
        principal_weights = centroids[
            cluster_id,
            principal_indices
        ]

        results_clusters.append({
            "cluster": cluster_id,
            "representative_tokens": ", ".join(principal_terms)
        })

    # Criar DataFrame
    df_tokens_clusters = pd.DataFrame(results_clusters)

    display(df_tokens_clusters)

----------------------------------------------------------------------------------------------------
BoW
----------------------------------------------------------------------------------------------------


,cluster,representative_tokens
0,0,"não, mas, bem, comida, muito, buffet, sem, ate..."
1,1,"restaurante, carne, r, não, ponto, pois, comid..."
2,2,"and, the, of, is, good, food, very, to, it, great"
3,3,"atendimento, comida, ótimo, excelente, ambient..."
4,4,"la, es, y, muy, carne, el, un, buena, lugar, c..."
5,5,"comida, excelente, boa, deliciosa, lugar, mara..."
6,6,"muito, comida, bem, atendimento, não, experiên..."
7,7,"excelente, top, melhor, lugar, maravilhoso, bl..."
8,8,"muito, comida, bom, atendimento, bem, boa, amb..."
9,9,"to, the, and, was, we, in, of, good, i, were"


----------------------------------------------------------------------------------------------------
TF-IDF
----------------------------------------------------------------------------------------------------


,cluster,representative_tokens
0,0,"muito, comida, saborosa, bem, bom, gostosa, lu..."
1,1,"comida, recomendo, lugar, não, qualidade, ambi..."
2,2,"deliciosa, comida, atendimento, experiência, s..."
3,3,"melhor, blumenau, restaurante, melhores, cidad..."
4,4,"maravilhoso, lugar, restaurante, tudo, atendim..."
5,5,"atendimento, ótimo, excelente, comida, ambient..."
6,6,"boa, comida, agradável, preço, local, acessíve..."
7,7,"excelente, comida, restaurante, tudo, recomend..."
8,8,"maravilhosa, comida, experiência, atendimento,..."
9,9,"ótimo, atendimento, lugar, local, tudo, almoço..."


----------------------------------------------------------------------------------------------------
Word2Vec
----------------------------------------------------------------------------------------------------
N/A
----------------------------------------------------------------------------------------------------
BERT
----------------------------------------------------------------------------------------------------
N/A


In [25]:
for model_name, data in model_data.items():
    print("-" * 100)
    print(model_name)
    print("-" * 100)

    # Centroides calculados pelo K-Means
    centroids = data["model"].cluster_centers_
    matrix = data["matrix"]

    # Similaridade entre avaliações e centroides
    similarities = cosine_similarity(matrix, centroids)

    # Número de avaliações retornadas
    n_reviews = 10

    cluster_column = f"cluster_{model_name.lower().replace('-', '_')}"

    for cluster_id in sorted(preprocessed_reviews_df[cluster_column].unique()):

        # Índices das avaliações pertencentes ao cluster
        cluster_indices = np.where(data["model"].labels_ == cluster_id)[0]

        if len(cluster_indices) == 0:
            continue

        # Similaridade de cada avaliação com o centroide do cluster
        cluster_similarities = similarities[cluster_indices, cluster_id]

        # Ordenar da maior para a menor similaridade ao centroide
        order = np.argsort(cluster_similarities)[::-1]
        ranked_indices = cluster_indices[order]

        selected_indices = []
        selected_texts = []
        selected_vectors = []

        for idx in ranked_indices:

            # Texto normalizado para evitar selecionar duplicatas exatas
            text = str(
                preprocessed_reviews_df.iloc[idx]["review_text"]
            ).strip().lower()

            # Evitar textos duplicados
            if text in selected_texts:
                continue

            candidate_vector = matrix[idx]

            # Verificar distância para as avaliações já selecionadas
            if selected_vectors:
                selected_similarity = cosine_similarity(
                    candidate_vector,
                    matrix[selected_indices]
                )[0]

                distances = 1 - selected_similarity

            selected_indices.append(idx)
            selected_texts.append(text)

            if len(selected_indices) >= n_reviews:
                break

        # Montar resultados
        cluster_df = preprocessed_reviews_df.iloc[
            selected_indices
        ].copy()

        cluster_df["similarity_to_centroid"] = similarities[
            selected_indices, cluster_id
        ]

        cluster_df["distance_to_centroid"] = (
            1 - cluster_df["similarity_to_centroid"]
        )

        print(f"\n{'=' * 100}")
        print(f"CLUSTER {cluster_id}")
        print("=" * 100)

        columns_to_show = [
            col for col in [
                "place_id",
                "rating",
                "similarity_to_centroid",
                "distance_to_centroid",
                "review_text"
            ]
            if col in cluster_df.columns
        ]

        display(cluster_df[columns_to_show].reset_index(drop=True))

----------------------------------------------------------------------------------------------------
BoW
----------------------------------------------------------------------------------------------------

CLUSTER 0


,place_id,rating,similarity_to_centroid,distance_to_centroid,review_text
0,0x94df1f216f80898f:0,4.0,0.573138,0.426862,"Comida boa. Mas não tem buffet livre, então nã..."
1,0x94df1fccc0cdf215:0,2.0,0.567096,0.432904,"O Tasty nunca foi um restaurante ótimo, foi se..."
2,0x94df1ee8c9ce17a5:0,2.0,0.540209,0.459791,Se você for depois da 13h já não vai pegar a m...
3,0x94df1973e56bf46d:0,1.0,0.536482,0.463518,"PÉSSIMO ATENDIMENTO!\nPrimeira vez que fomos, ..."
4,0x94df18eac4b2643d:0,5.0,0.512585,0.487415,"Lugar muito aconchegante, feijoada uma delícia..."
5,0x94df19733e66ed3f:0,2.0,0.508492,0.491508,"Não sou de avaliar lugares, más esse não teve ..."
6,0x94df192d88bb4431:0,5.0,0.503388,0.496612,Um local seguro para pessoas com alimentação r...
7,0x94defd80e5de4f0f:0,2.0,0.474330,0.525670,Não sei na outras empresas mas já foi servido...
8,0x94df18eb055b337f:0,1.0,0.467286,0.532714,"Minha experiência não foi das melhores, buffet..."
9,0x94df19f03cf0d20f:0,1.0,0.461327,0.538673,"Péssima experiência, comida é sem gosto e mal ..."



CLUSTER 1


,place_id,rating,similarity_to_centroid,distance_to_centroid,review_text
0,0x94df192bbbd5d5bb:0,5.0,1.0,-3.774758e-15,Melhor comida que já tocou meus lábios e que t...



CLUSTER 2


...

,place_id,rating,similarity_to_centroid,distance_to_centroid,review_text
0,0x94df18eac4b2643d:0,5.0,0.941216,0.058784,Ótimo atendimento!
1,0x94df1ee19d73e25d:0,5.0,0.939757,0.060243,Ótimo atendimento.
2,0x94df18c387e4e047:0,5.0,0.935020,0.064980,Atendimento excelente!
3,0x94df192d88bb4431:0,5.0,0.934241,0.065759,Ótimo atendimento e comida
4,0x94df18bfe8d9f45f:0,5.0,0.928943,0.071057,"Ótimo atendimento,comida excelente ."
5,0x94df18eb055b337f:0,5.0,0.926379,0.073621,Lugar excelente com atendimento imperial!
6,0x94df1fccc0cdf215:0,5.0,0.923074,0.076926,Excelente restaurante e ótimo atendimento..
7,0x94df19733e66ed3f:0,5.0,0.922117,0.077883,Ótima comida e atendimento.
8,0x94df192acbd036eb:0,5.0,0.919356,0.080644,Exelente restaurante
9,0x94df1ee19d73e25d:0,4.0,0.919172,0.080828,Otimo estabelecimento.



CLUSTER 9


,place_id,rating,similarity_to_centroid,distance_to_centroid,review_text
0,0x94df192d88bb4431:0,5.0,0.940286,0.059714,Amazing food. Great variety of gluten and lact...
1,0x94df198ab08528b1:0,5.0,0.939111,0.060889,Great place for a fine dining. Expect to spend...
2,0x94df1f708192624b:0,5.0,0.938701,0.061299,Excellent food. German beer and own brewery. N...
3,0x94df1ece1bfad0c7:0,5.0,0.937528,0.062472,One of the best idea of fast food ever perform...
4,0x94df1f708192624b:0,5.0,0.937183,0.062817,A little Germany in the middle of Blumenau. Gr...
5,0x94df1f708192624b:0,5.0,0.935689,0.064311,The best German place that I was\nVery nice ex...
6,0x94df198ab08528b1:0,4.0,0.934301,0.065699,"Nice place! Food is pretty good, music is alri..."
7,0x94df18bfe8d9f45f:0,4.0,0.933128,0.066872,"A very nice place, very peaceful! I have lunch..."
8,0x94df192acbd036eb:0,5.0,0.932854,0.067146,"Excelent food, good enviroment, fair price (bu..."
9,0x94df192acbd036eb:0,5.0,0.932412,0.067588,"A very good place to have lunch, with tasty fo..."



CLUSTER 10


,place_id,rating,similarity_to_centroid,distance_to_centroid,review_text
0,0x94df1faf33d74b45:0,5.0,0.948233,0.051767,"Ambiente organizado, comida saborosa e atendim..."
1,0x94df1f925f65034b:0,5.0,0.938654,0.061346,Comida muito saborosa e ótimo atendimento.
2,0x94df195aa9f871a3:0,5.0,0.937183,0.062817,"Comida bem preparada, gostosa e prática. O ate..."
3,0x94df1973e56bf46d:0,5.0,0.935366,0.064634,"Excelente, atendimento excepcional e a comida ..."
4,0x94df1ece1bfad0c7:0,5.0,0.933515,0.066485,Comida muito boa e ótimo atendimento.
5,0x94df1956fa84d84f:0,5.0,0.932464,0.067536,"Comida maravilhosa, preço acessível e atendime..."
6,0x94df1ece1bfad0c7:0,5.0,0.932449,0.067551,Que salada incrível!! Muito saborosa. Atendime...
7,0x94df18c387e4e047:0,5.0,0.932045,0.067955,Atendimento muito bom e comida deliciosa!!
8,0x94df1ece1bfad0c7:0,5.0,0.931971,0.068029,Ótimo atendimento e comida maravilhosa!
9,0x94df1975c038b80f:0,5.0,0.931022,0.068978,"Comida sempre fresca, sempre saborosa.\nAmbien..."


## Coesão interna

In [26]:
for model_name, data in model_data.items():
    print("-" * 100)
    print(model_name)
    print("-" * 100)

    results_cohesion = []
    cluster_column = f"cluster_{model_name.lower().replace('-', '_')}"

    for cluster_id in sorted(preprocessed_reviews_df[cluster_column].unique()):

        # Índices dos documentos pertencentes ao cluster
        indices = preprocessed_reviews_df.index[
                preprocessed_reviews_df[cluster_column] == cluster_id
            ].tolist()

        # Extrair a submatriz de similaridade
        submatrix = data["similarity"][np.ix_(indices, indices)]

        # Selecionar apenas a parte superior da matriz,
        # excluindo a diagonal principal
        even_values = submatrix[
            np.triu_indices_from(submatrix, k=1)
        ]

        # Calcular a similaridade média
        if len(even_values) > 0:
            mean_cohesion = even_values.mean()
        else:
            mean_cohesion = 1.0

        results_cohesion.append({
            "cluster": cluster_id,
            "n_documents": len(indices),
            "mean_cohesion": mean_cohesion
        })


    df_cohesion = pd.DataFrame(results_cohesion)

    display(df_cohesion)

----------------------------------------------------------------------------------------------------
BoW
----------------------------------------------------------------------------------------------------


,cluster,n_documents,mean_cohesion
0,0,46,0.137046
1,1,1,1.000000
2,2,16,0.308375
3,3,277,0.246987
4,4,1,1.000000
5,5,352,0.301546
6,6,15,0.226144
7,7,1174,0.029567
8,8,378,0.291574
9,9,2,0.464581


----------------------------------------------------------------------------------------------------
TF-IDF
----------------------------------------------------------------------------------------------------


,cluster,n_documents,mean_cohesion
0,0,264,0.082605
1,1,997,0.010289
2,2,40,0.744782
3,3,91,0.275865
4,4,67,0.834591
5,5,168,0.128897
6,6,50,0.601422
7,7,152,0.702613
8,8,49,0.438041
9,9,58,0.680060


----------------------------------------------------------------------------------------------------
Word2Vec
----------------------------------------------------------------------------------------------------


,cluster,n_documents,mean_cohesion
0,0,136,0.644392
1,1,1286,0.429267
2,2,65,0.900310
3,3,108,0.834920
4,4,89,0.927012
5,5,22,1.000000
6,6,147,0.815738
7,7,351,0.561836
8,8,58,0.816658


----------------------------------------------------------------------------------------------------
BERT
----------------------------------------------------------------------------------------------------


,cluster,n_documents,mean_cohesion
0,0,58,0.786133
1,1,245,0.794135
2,2,248,0.696257
3,3,297,0.735045
4,4,188,0.533684
5,5,70,0.947932
6,6,29,0.775738
7,7,416,0.732918
8,8,182,0.734749
9,9,99,0.715527


## Distância entre clusters

In [27]:
for model_name, data in model_data.items():
    print("-" * 100)
    print(model_name)
    print("-" * 100)

    # Similaridade entre os centróides
    similarity_centroids = cosine_similarity(
        data["model"].cluster_centers_
    )

    # Converter similaridade em distância
    distance_centroids = 1 - similarity_centroids

    data["distance_centroids"] = distance_centroids

    # Criar DataFrame
    df_dist_clusters = pd.DataFrame(
        distance_centroids,
        index=[f"Cluster {i}" for i in range(data["k"])],
        columns=[f"Cluster {i}" for i in range(data["k"])]
    )

    display(df_dist_clusters.round(3))

----------------------------------------------------------------------------------------------------
BoW
----------------------------------------------------------------------------------------------------


,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7,Cluster 8,Cluster 9
Cluster 0,0.000,0.504,0.993,0.638,0.975,0.664,0.404,0.620,0.605,1.000
Cluster 1,0.504,0.000,0.997,0.770,0.929,0.766,0.633,0.705,0.743,0.994
Cluster 2,0.993,0.997,-0.000,0.994,1.000,0.997,0.989,0.805,0.983,0.339
Cluster 3,0.638,0.770,0.994,0.000,0.986,0.589,0.579,0.614,0.594,1.000
Cluster 4,0.975,0.929,1.000,0.986,0.000,0.941,0.992,0.906,0.987,1.000
Cluster 5,0.664,0.766,0.997,0.589,0.941,-0.000,0.644,0.748,0.627,1.000
Cluster 6,0.404,0.633,0.989,0.579,0.992,0.644,0.000,0.701,0.198,1.000
Cluster 7,0.620,0.705,0.805,0.614,0.906,0.748,0.701,0.000,0.767,0.870
Cluster 8,0.605,0.743,0.983,0.594,0.987,0.627,0.198,0.767,0.000,1.000
Cluster 9,1.000,0.994,0.339,1.000,1.000,1.000,1.000,0.870,1.000,0.000


----------------------------------------------------------------------------------------------------
TF-IDF
----------------------------------------------------------------------------------------------------


,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7,Cluster 8,Cluster 9,Cluster 10,Cluster 11,Cluster 12,Cluster 13,Cluster 14
Cluster 0,-0.000,0.482,0.749,0.924,0.954,0.567,0.767,0.842,0.790,0.908,0.556,0.931,0.449,0.959,0.838
Cluster 1,0.482,-0.000,0.794,0.797,0.933,0.567,0.794,0.797,0.825,0.857,0.870,0.924,0.771,0.950,0.836
Cluster 2,0.749,0.794,0.000,0.984,0.997,0.840,0.833,0.928,0.870,0.952,0.951,0.970,0.821,0.986,0.862
Cluster 3,0.924,0.797,0.984,0.000,0.979,0.935,0.971,0.970,0.975,0.986,0.984,0.991,0.969,0.983,0.985
Cluster 4,0.954,0.933,0.997,0.979,0.000,0.915,0.998,0.994,0.996,0.988,0.996,0.997,0.998,0.986,0.996
Cluster 5,0.567,0.567,0.840,0.935,0.915,0.000,0.845,0.674,0.832,0.519,0.796,0.911,0.763,0.960,0.837
Cluster 6,0.767,0.794,0.833,0.971,0.998,0.845,-0.000,0.940,0.891,0.974,0.973,0.976,0.181,0.989,0.893
Cluster 7,0.842,0.797,0.928,0.970,0.994,0.674,0.940,0.000,0.918,0.984,0.994,0.971,0.920,0.993,0.942
Cluster 8,0.790,0.825,0.870,0.975,0.996,0.832,0.891,0.918,-0.000,0.962,0.972,0.946,0.879,0.977,0.881
Cluster 9,0.908,0.857,0.952,0.986,0.988,0.519,0.974,0.984,0.962,0.000,0.978,0.971,0.963,0.992,0.966


----------------------------------------------------------------------------------------------------
Word2Vec
----------------------------------------------------------------------------------------------------


,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7,Cluster 8
Cluster 0,0.000,0.453,0.747,0.826,0.654,0.782,0.829,0.674,0.806
Cluster 1,0.453,0.000,0.551,0.445,0.542,0.597,0.499,0.203,0.482
Cluster 2,0.747,0.551,-0.000,0.745,0.708,0.747,0.783,0.638,0.692
Cluster 3,0.826,0.445,0.745,-0.000,0.713,0.902,0.721,0.468,0.668
Cluster 4,0.654,0.542,0.708,0.713,0.000,0.760,0.794,0.635,0.840
Cluster 5,0.782,0.597,0.747,0.902,0.760,0.000,0.858,0.734,0.779
Cluster 6,0.829,0.499,0.783,0.721,0.794,0.858,0.000,0.548,0.747
Cluster 7,0.674,0.203,0.638,0.468,0.635,0.734,0.548,-0.000,0.589
Cluster 8,0.806,0.482,0.692,0.668,0.840,0.779,0.747,0.589,0.000


----------------------------------------------------------------------------------------------------
BERT
----------------------------------------------------------------------------------------------------


,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7,Cluster 8,Cluster 9,Cluster 10
Cluster 0,0.000,0.384,0.294,0.224,0.306,0.529,0.433,0.260,0.250,0.453,0.237
Cluster 1,0.384,-0.000,0.130,0.283,0.134,0.280,0.549,0.480,0.175,0.586,0.355
Cluster 2,0.294,0.130,0.000,0.201,0.183,0.417,0.445,0.282,0.155,0.509,0.197
Cluster 3,0.224,0.283,0.201,0.000,0.253,0.488,0.431,0.214,0.144,0.495,0.125
Cluster 4,0.306,0.134,0.183,0.253,0.000,0.187,0.485,0.449,0.242,0.414,0.387
Cluster 5,0.529,0.280,0.417,0.488,0.187,-0.000,0.697,0.741,0.473,0.622,0.663
Cluster 6,0.433,0.549,0.445,0.431,0.485,0.697,-0.000,0.346,0.433,0.388,0.369
Cluster 7,0.260,0.480,0.282,0.214,0.449,0.741,0.346,0.000,0.246,0.407,0.069
Cluster 8,0.250,0.175,0.155,0.144,0.242,0.473,0.433,0.246,0.000,0.527,0.098
Cluster 9,0.453,0.586,0.509,0.495,0.414,0.622,0.388,0.407,0.527,0.000,0.474


In [28]:
for model_name, data in model_data.items():
    print("-" * 100)
    print(model_name)
    print("-" * 100)

    results_distances = []

    for i in range(data["k"]):
        for j in range(i + 1, data["k"]):

            results_distances.append({
                "cluster_1": i,
                "cluster_2": j,
                "distance": data["distance_centroids"][i, j]
            })

    df_pairs_clusters = pd.DataFrame(
        results_distances
    )

    display(df_pairs_clusters.sort_values("distance"))

    # Identificar menor e maior distância entre clusters
    nearest_pair = df_pairs_clusters.loc[
        df_pairs_clusters["distance"].idxmin()
    ]

    furthest_pair = df_pairs_clusters.loc[
        df_pairs_clusters["distance"].idxmax()
    ]

    # Clusters mais próximos
    c1_prox = int(nearest_pair["cluster_1"])
    c2_prox = int(nearest_pair["cluster_2"])

    # Clusters mais distantes
    c1_dist = int(furthest_pair["cluster_1"])
    c2_dist = int(furthest_pair["cluster_2"])


    print("CLUSTERS MAIS PRÓXIMOS")
    print("----------------------")
    print(
        f"Cluster {c1_prox} × Cluster {c2_prox}"
    )
    print(
        f"Distância: {nearest_pair['distance']:.3f}"
    )


    print("\nCLUSTERS MAIS DISTANTES")
    print("-----------------------")
    print(
        f"Cluster {c1_dist} × Cluster {c2_dist}"
    )
    print(
        f"Distância: {furthest_pair['distance']:.3f}"
    )

----------------------------------------------------------------------------------------------------
BoW
----------------------------------------------------------------------------------------------------


,cluster_1,cluster_2,distance
40,6,8,0.197784
23,2,9,0.338862
5,0,6,0.404232
0,0,1,0.503767
26,3,6,0.578747
25,3,5,0.588690
28,3,8,0.593669
7,0,8,0.604511
27,3,7,0.613684
6,0,7,0.619583


CLUSTERS MAIS PRÓXIMOS
----------------------
Cluster 6 × Cluster 8
Distância: 0.198

CLUSTERS MAIS DISTANTES
-----------------------
Cluster 2 × Cluster 4
Distância: 1.000
----------------------------------------------------------------------------------------------------
TF-IDF
----------------------------------------------------------------------------------------------------


,cluster_1,cluster_2,distance
74,6,12,0.181068
11,0,12,0.448742
0,0,1,0.482146
63,5,9,0.518516
9,0,10,0.556187
...,...,...,...
55,4,10,0.996189
56,4,11,0.996520
28,2,4,0.997029
57,4,12,0.997949


CLUSTERS MAIS PRÓXIMOS
----------------------
Cluster 6 × Cluster 12
Distância: 0.181

CLUSTERS MAIS DISTANTES
-----------------------
Cluster 4 × Cluster 6
Distância: 0.998
----------------------------------------------------------------------------------------------------
Word2Vec
----------------------------------------------------------------------------------------------------


,cluster_1,cluster_2,distance
13,1,7,0.202575
9,1,3,0.444811
0,0,1,0.452895
24,3,7,0.467507
14,1,8,0.482138
12,1,6,0.499041
10,1,4,0.542148
33,6,7,0.548289
8,1,2,0.550609
35,7,8,0.589034


CLUSTERS MAIS PRÓXIMOS
----------------------
Cluster 1 × Cluster 7
Distância: 0.203

CLUSTERS MAIS DISTANTES
-----------------------
Cluster 3 × Cluster 5
Distância: 0.902
----------------------------------------------------------------------------------------------------
BERT
----------------------------------------------------------------------------------------------------


,cluster_1,cluster_2,distance
51,7,10,0.068591
53,8,10,0.097845
33,3,10,0.125267
10,1,2,0.129535
12,1,4,0.133730
31,3,8,0.144437
24,2,8,0.155491
16,1,8,0.175212
20,2,4,0.182846
34,4,5,0.186649


CLUSTERS MAIS PRÓXIMOS
----------------------
Cluster 7 × Cluster 10
Distância: 0.069

CLUSTERS MAIS DISTANTES
-----------------------
Cluster 5 × Cluster 7
Distância: 0.741
